# SegFormer windowpane 감지 테스트 (Colab)

ADE20K로 학습된 SegFormer **B0 / B2 / B5**가 실내 사진에서 `windowpane` 클래스를 얼마나 잘 잡는지 비교합니다.
OneFormer 테스트와 같은 이미지·같은 지표(면적 비율, 평균 확률, 연결요소 박스 수, 추론 시간)를 씁니다.

- **GPU로 빠르게 비교**: `런타임 → 런타임 유형 변경 → T4 GPU`
- **CPU 속도를 재고 싶을 때**: 설정 셀에서 `FORCE_CPU = True` (Colab CPU는 2코어라 일반 PC보다 느릴 수 있음)

In [ ]:
# 1. 패키지 설치
!pip -q install -U transformers pillow-avif-plugin

In [ ]:
# 2. 임포트와 설정
import os, glob, time, shutil
import numpy as np
import torch
import torch.nn.functional as F
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
try:
    import pillow_avif  # noqa: F401
except ImportError:
    pass
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

MODELS = {
    "B0": "nvidia/segformer-b0-finetuned-ade-512-512",
    "B2": "nvidia/segformer-b2-finetuned-ade-512-512",
    "B5": "nvidia/segformer-b5-finetuned-ade-640-640",
}
IMG_DIR = "window_test_pics"
OUT_DIR = "segformer_test_results"
MAX_SIDE = 1024        # 원본 긴 변이 이보다 크면 축소 (결과 해상도 기준)
KEEP_ASPECT = False    # False: 공식 방식(정사각형으로 늘려 입력) / True: 비율 유지, 짧은 변을 모델 크기에 맞춤
PROB_THR = 0.3         # 커튼 뒤 창문처럼 argmax에서 밀린 영역을 보기 위한 확률 기준
CC_MIN_AREA = 0.001    # 연결요소 최소 면적(이미지 대비 비율)
EXTS = (".jpg", ".jpeg", ".png", ".avif", ".webp")
CONFUSABLE = ["door", "curtain", "mirror", "screen door", "blind", "painting"]
FORCE_CPU = False

device = "cpu" if FORCE_CPU or not torch.cuda.is_available() else "cuda"
if device == "cpu":
    torch.set_num_threads(os.cpu_count())
print("device:", device, "| threads:", torch.get_num_threads())
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 3. 테스트 이미지 업로드 (폴더가 비어 있으면 업로드 창이 뜸)
# Drive 사용 시: from google.colab import drive; drive.mount('/content/drive'); IMG_DIR = '/content/drive/MyDrive/window_test_pics'
from google.colab import files

if not [f for f in os.listdir(IMG_DIR) if f.lower().endswith(EXTS)]:
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(IMG_DIR, name), "wb") as f:
            f.write(data)
        if os.path.exists(name):
            os.remove(name)

img_paths = sorted(p for p in glob.glob(os.path.join(IMG_DIR, "*")) if p.lower().endswith(EXTS))
print(f"{len(img_paths)}장:", [os.path.basename(p) for p in img_paths])

In [ ]:
# 4. 모델 로드
def find_label(id2label, name):
    for i, l in id2label.items():
        if l == name:
            return i
    for i, l in id2label.items():
        if l.startswith(name):
            return i
    return None

models = {}
for key, mid in MODELS.items():
    t0 = time.perf_counter()
    proc = SegformerImageProcessor.from_pretrained(mid)
    model = SegformerForSemanticSegmentation.from_pretrained(mid).to(device).eval()
    id2label = {int(k): v.strip() for k, v in model.config.id2label.items()}
    win = find_label(id2label, "windowpane")
    if win is None:
        win = find_label(id2label, "window")
    assert win is not None, f"{key}: windowpane 클래스를 찾지 못함"
    conf = {n: find_label(id2label, n) for n in CONFUSABLE if find_label(id2label, n) is not None}
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    side = proc.size.get("height", 512)
    models[key] = dict(proc=proc, model=model, win=win, conf=conf, side=side)
    print(f"{key}: {n_params:.1f}M params | input {side} | windowpane id={win} "
          f"({id2label[win]!r}) | load {time.perf_counter() - t0:.1f}s")

In [ ]:
# 5. 함수 정의
def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    s = MAX_SIDE / max(w, h)
    if s < 1:
        img = img.resize((round(w * s), round(h * s)), Image.LANCZOS)
    return img

def prepare(img, m):
    if KEEP_ASPECT:
        s = m["side"] / min(img.size)
        im = img.resize((max(32, round(img.width * s)), max(32, round(img.height * s))), Image.BILINEAR)
        return m["proc"](images=im, do_resize=False, return_tensors="pt")
    return m["proc"](images=img, return_tensors="pt")

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def run_model(img, m):
    inputs = prepare(img, m).to(device)
    sync(); t0 = time.perf_counter()
    logits = m["model"](**inputs).logits            # (1, C, h/4, w/4)
    sync(); t_model = time.perf_counter() - t0
    logits = F.interpolate(logits, size=(img.height, img.width), mode="bilinear", align_corners=False)[0]
    argmax = logits.argmax(0).cpu().numpy()
    win_prob = logits.softmax(0)[m["win"]].cpu().numpy()
    return argmax, win_prob, t_model

def cc_boxes(mask):
    H, W = mask.shape
    _, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    return [(x, y, x + w, y + h) for x, y, w, h, a in stats[1:] if a >= CC_MIN_AREA * H * W]

def draw_boxes(ax, boxes, color):
    for x0, y0, x1, y1 in boxes:
        ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=color, lw=2))

def overlay(arr, mask, thr_mask):
    out = arr.copy()
    out[mask] = (0.45 * out[mask] + 0.55 * np.array([255, 0, 0])).astype(np.uint8)
    contours, _ = cv2.findContours(thr_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (0, 255, 255), 2)   # 시안 = prob >= PROB_THR
    return out

In [ ]:
# 6. 실행
# 모델마다 한 번씩 워밍업 (첫 추론은 초기화 때문에 느림)
if img_paths:
    w_img = load_image(img_paths[0])
    for m in models.values():
        run_model(w_img, m)

rows = []
for p in img_paths:
    name = os.path.splitext(os.path.basename(p))[0]
    try:
        img = load_image(p)
    except Exception as e:
        print(f"[skip] {p}: {e}")
        continue
    arr = np.asarray(img)

    n = len(models)
    fig, axes = plt.subplots(2, n + 1, figsize=(5.5 * (n + 1), 9))
    axes[0, 0].imshow(arr); axes[0, 0].set_title("Original")
    axes[1, 0].axis("off")
    axes[1, 0].text(0.02, 0.6, "Row 1: red = argmax windowpane\n"
                    f"        cyan = prob >= {PROB_THR}\n        yellow = CC boxes\n"
                    "Row 2: windowpane probability", fontsize=10, family="monospace")

    for j, (key, m) in enumerate(models.items(), start=1):
        argmax, win_prob, t_model = run_model(img, m)
        mask = argmax == m["win"]
        thr_mask = win_prob >= PROB_THR
        cc = cc_boxes(mask)
        cc_thr = cc_boxes(thr_mask)

        axes[0, j].imshow(overlay(arr, mask, thr_mask)); draw_boxes(axes[0, j], cc, "yellow")
        axes[0, j].set_title(f"{key}: {len(cc)} boxes | {t_model:.2f}s")
        im = axes[1, j].imshow(win_prob, cmap="inferno", vmin=0, vmax=1)
        axes[1, j].set_title(f"{key} windowpane prob")

        row = {
            "image": os.path.basename(p),
            "model": key,
            "window_%": round(100 * mask.mean(), 2),
            f"window_%(p>={PROB_THR})": round(100 * thr_mask.mean(), 2),
            "window_mean_prob": round(float(win_prob[mask].mean()), 3) if mask.any() else np.nan,
            "cc_boxes": len(cc),
            f"cc_boxes(p>={PROB_THR})": len(cc_thr),
            "t_model_s": round(t_model, 3),
        }
        for cname, cid in m["conf"].items():
            row[f"{cname}_%"] = round(100 * (argmax == cid).mean(), 2)
        rows.append(row)

    for a in axes.ravel():
        a.set_xticks([]); a.set_yticks([])
    fig.colorbar(im, ax=axes[1, 1:].tolist(), fraction=0.02)
    fig.suptitle(f"{name}  ({'keep aspect' if KEEP_ASPECT else 'square resize'}, {device})")
    fig.savefig(os.path.join(OUT_DIR, f"{name}_segformer.png"), dpi=100, bbox_inches="tight")
    plt.show()
    plt.close(fig)

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False, encoding="utf-8-sig")
df

In [ ]:
# 7. 모델별 평균 (추론 시간, 창문 면적, 박스 수)
df.groupby("model")[["t_model_s", "window_%", f"window_%(p>={PROB_THR})", "cc_boxes"]].mean().round(3)

In [ ]:
# 8. 결과 다운로드 (비교 이미지 + summary.csv)
shutil.make_archive("segformer_test_results", "zip", OUT_DIR)
files.download("segformer_test_results.zip")